# truck-n-trailer

<a href="https://colab.research.google.com/github/ericbreh/truck-n-trailer/blob/main/truck-n-trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
  !pip install polytope
  !pip install -q pyomo
  !apt-get install -y -qq glpk-utils
  !apt-get install -y -qq coinor-cbc
  !wget -N -q "https://matematica.unipv.it/gualandi/solvers/ipopt-linux64.zip"
  !unzip -o -q ipopt-linux64

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 150
import pyomo.environ as pyo
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

### Dynamics

In [ ]:
def truck_trailer_dynamics(q, u, L, d):
    # q = [x, y, theta_t, theta_l, v, phi]
    # u = [a, phi_dot]
    x, y, theta_t, theta_l, v, phi = q
    a, phi_dot = u

    x_dot = v * np.cos(theta_t)
    y_dot = v * np.sin(theta_t)
    theta_t_dot = (v / L) * np.tan(phi)
    theta_l_dot = (v / d) * np.sin(theta_t - theta_l)
    v_dot = a
    phi_dot = phi_dot

    return np.array([x_dot, y_dot, theta_t_dot, theta_l_dot, v_dot, phi_dot])


### Pyomo Model

In [ ]:
def build_pyomo_model(params):
    N = params["N"]
    dt = params["dt"]
    L = params["L"]
    d = params["d"]
    target = params["target"]

    # Control bounds
    a_min = params.get("a_min", -2.0)
    a_max = params.get("a_max", 2.0)
    phi_dot_min = params.get("phi_dot_min", -1.0)
    phi_dot_max = params.get("phi_dot_max", 1.0)

    # State bounds
    v_min = params.get("v_min", 0.0)
    v_max = params.get("v_max", 3.0)
    phi_min = params.get("phi_min", -1.5)
    phi_max = params.get("phi_max", 1.5)

    w_control = params.get("w_control", 0.01)
    max_jackknife = params.get("max_jackknife_angle", np.pi/2)

    model = pyo.ConcreteModel()

    # Sets
    model.T = pyo.RangeSet(0, N)
    model.K = pyo.RangeSet(0, N - 1)
    model.S = pyo.RangeSet(0, 5)  # x, y, theta_t, theta_l, v, phi

    # State variables
    model.x = pyo.Var(model.S, model.T, domain=pyo.Reals)

    # Control variables
    model.a = pyo.Var(model.K, bounds=(a_min, a_max))
    model.phi_dot = pyo.Var(model.K, bounds=(phi_dot_min, phi_dot_max))

    # Dynamics
    def dyn_rule(model, s, k):
        if s == 0:  # x
            return model.x[0, k+1] == model.x[0, k] + dt * (model.x[4, k] * pyo.cos(model.x[2, k]))
        if s == 1:  # y
            return model.x[1, k+1] == model.x[1, k] + dt * (model.x[4, k] * pyo.sin(model.x[2, k]))
        if s == 2:  # theta_t
            return model.x[2, k+1] == model.x[2, k] + dt * ((model.x[4, k] / L) * pyo.tan(model.x[5, k]))
        if s == 3:  # theta_l
            return model.x[3, k+1] == model.x[3, k] + dt * ((model.x[4, k] / d) * pyo.sin(model.x[2, k] - model.x[3, k]))
        if s == 4:  # v
            return model.x[4, k+1] == model.x[4, k] + dt * model.a[k]
        if s == 5:  # phi
            return model.x[5, k+1] == model.x[5, k] + dt * model.phi_dot[k]

    model.dyn = pyo.Constraint(model.S, model.K, rule=dyn_rule)

    # Jackknife constraint
    def jackknife_rule(model, t):
        return pyo.inequality(-max_jackknife, model.x[2, t] - model.x[3, t], max_jackknife)
    model.jackknife_cons = pyo.Constraint(model.T, rule=jackknife_rule)

    # Velocity & steering angle constraints
    model.vel_cons = pyo.Constraint(model.T, rule=lambda m, t: pyo.inequality(v_min, m.x[4, t], v_max))
    model.phi_cons = pyo.Constraint(model.T, rule=lambda m, t: pyo.inequality(phi_min, m.x[5, t], phi_max))

    # Objective
    def obj_rule(model):
        final_err = sum((model.x[i, N] - target[i])**2 for i in range(4))
        control_effort = sum(model.a[k]**2 + model.phi_dot[k]**2 for k in model.K)
        return final_err + w_control * control_effort
    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    return model


### MPC

In [ ]:
def set_initial_guess(model, u_guess):
    N = u_guess.shape[1]
    for k in range(N):
        model.a[k].set_value(float(u_guess[0, k]))
        model.phi_dot[k].set_value(float(u_guess[1, k]))

def extract_solution(model, params):
    N = params["N"]
    x_opt = np.zeros((6, N+1))
    u_opt = np.zeros((2, N))
    for i in range(6):
        for t in range(N+1):
            x_opt[i, t] = pyo.value(model.x[i, t])
    for k in range(N):
        u_opt[0, k] = pyo.value(model.a[k])
        u_opt[1, k] = pyo.value(model.phi_dot[k])
    return x_opt, u_opt


def run_mpc_loop(params):
    waypoints = params["waypoints"]
    waypoint_tol = params.get("waypoint_tol", 0.5)

    if len(waypoints) == 0:
        raise ValueError("waypoints list cannot be empty")

    # Initialize tracking
    q_history_full = []
    u_history_full = []
    waypoint_switches = [0]  # Start at index 0

    q_current = params["q0"].copy()
    N = params["N"]

    # Initial control guess
    u_guess = np.zeros((2, N))
    u_guess[0, :] = params.get("a_init", 0.0)
    u_guess[1, :] = params.get("phi_dot_init", 0.0)

    q_history_full.append(q_current.copy())

    print(
        f"Starting waypoint navigation through {len(waypoints)} waypoints...")

    # Navigate through each waypoint
    for wp_idx, waypoint in enumerate(waypoints):
        print(
            f"\n--- Waypoint {wp_idx + 1}/{len(waypoints)}: ({waypoint[0]:.1f}, {waypoint[1]:.1f}) ---")

        # Update target in params for this waypoint
        params_wp = params.copy()
        params_wp["target"] = waypoint

        # Build model with new target
        model = build_pyomo_model(params_wp)
        solver = pyo.SolverFactory("ipopt")
        solver.options["max_iter"] = params.get("solver_max_iter", 500)
        solver.options["tol"] = params.get("solver_tol", 1e-6)
        solver.options["print_level"] = params.get("solver_print_level", 0)

        # Fix initial state for this waypoint segment
        for i in range(6):
            model.x[i, 0].fix(float(q_current[i]))

        step = 0
        max_steps_per_waypoint = params.get("max_steps", 200)

        # Navigate to current waypoint
        while np.linalg.norm(q_current[:2] - waypoint[:2]) > waypoint_tol and step < max_steps_per_waypoint:
            # Update initial state
            for i in range(6):
                model.x[i, 0].fix(float(q_current[i]))

            # Warm start
            set_initial_guess(model, u_guess)

            # Solve
            result = solver.solve(model, tee=False)
            term = result.solver.termination_condition
            if term not in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.locallyOptimal):
                print(f"  Solver warning: {term} at step {step}")
                break

            # Extract solution
            x_opt, u_opt = extract_solution(model, params_wp)

            # Simulate one step
            u_star = u_opt[:, 0]
            q_dot = truck_trailer_dynamics(
                q_current, u_star, params["L"], params["d"])
            q_next = q_current + params["dt"] * q_dot

            # Record
            q_history_full.append(q_next.copy())
            u_history_full.append(u_star.copy())

            # Update
            q_current = q_next.copy()
            step += 1

            # Shift control horizon
            u_guess = np.hstack([u_opt[:, 1:], u_opt[:, -1:]])

        # Record waypoint switch
        waypoint_switches.append(len(q_history_full) - 1)

        dist_to_waypoint = np.linalg.norm(q_current[:2] - waypoint[:2])
        print(
            f"  Reached waypoint in {step} steps. Final error: {dist_to_waypoint:.3f} m")

        if step >= max_steps_per_waypoint:
            print(f"  Warning: Max steps reached before reaching waypoint")

    print(
        f"\nWaypoint navigation complete! Total steps: {len(q_history_full) - 1}")

    return np.array(q_history_full), np.array(u_history_full), waypoint_switches

### Simulation

In [ ]:
def plot_trajectory(q_hist, u_hist, params):
    q_hist = np.atleast_2d(q_hist)
    if q_hist.shape[0] == 6:
        q_hist = q_hist.T

    fig, axes = plt.subplots(3, 2, figsize=(14, 12))

    # States
    state_labels = ["x (m)", "y (m)", "theta_t (rad)", "theta_l (rad)", "v (m/s)", "phi (rad)"]
    for i, ax in enumerate(axes.flatten()):
        if i < 6:
            ax.plot(np.arange(q_hist.shape[0])*params["dt"], q_hist[:, i], "-o", markersize=3)
            ax.set_title(state_labels[i])
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Controls
    if len(u_hist) > 0:
        t = np.arange(len(u_hist))*params["dt"]
        plt.figure(figsize=(10,4))
        plt.step(t, u_hist[:, 0], where="post", label="a (m/s²)")
        plt.step(t, u_hist[:, 1], where="post", label="phi_dot (rad/s)")
        plt.xlabel("Time (s)")
        plt.ylabel("Control")
        plt.title("Applied Controls")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()


def animate_waypoint_trajectory(q_hist, params, waypoint_switches, interval=80):
    q_hist = np.atleast_2d(q_hist)
    if q_hist.shape[0] == 6:
        q_hist = q_hist.T

    L = params["L"]
    d = params["d"]
    M = len(q_hist)
    waypoints = params["waypoints"]

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("x (m)", fontsize=12)
    ax.set_ylabel("y (m)", fontsize=12)
    ax.set_title("Path Following", fontsize=14, fontweight='bold')

    # Calculate axis limits
    all_x = np.concatenate([q_hist[:, 0], [wp[0] for wp in waypoints]])
    all_y = np.concatenate([q_hist[:, 1], [wp[1] for wp in waypoints]])
    margin = max(8.0, 0.2 * max(np.ptp(all_x), np.ptp(all_y)))
    ax.set_xlim(all_x.min() - margin, all_x.max() + margin)
    ax.set_ylim(all_y.min() - margin, all_y.max() + margin)

    # Plot full path
    ax.plot(q_hist[:, 0], q_hist[:, 1], "k--",
            alpha=0.3, lw=1.5, zorder=1)

    # Plot waypoints
    for i, wp in enumerate(waypoints):
        ax.plot(wp[0], wp[1], "go", ms=12, mew=2, markeredgecolor='darkgreen',
                alpha=0.7, zorder=8)

    # Animation artists
    truck_line, = ax.plot([], [], "b-", lw=5,
                          label="Truck", solid_capstyle="round")
    trailer_line, = ax.plot(
        [], [], "r-", lw=5, label="Trailer", solid_capstyle="round")
    truck_pts, = ax.plot([], [], "bo", ms=8)
    trailer_pts, = ax.plot([], [], "ro", ms=8)
    hitch_pt, = ax.plot([], [], "ko", ms=6)

    def init():
        truck_line.set_data([], [])
        trailer_line.set_data([], [])
        truck_pts.set_data([], [])
        trailer_pts.set_data([], [])
        hitch_pt.set_data([], [])
        return truck_line, trailer_line, truck_pts, trailer_pts, hitch_pt

    def update(frame):
        x, y, theta_t, theta_l = q_hist[frame, :4]

        # Truck: from rear axle to front axle
        x_front = x + L * np.cos(theta_t)
        y_front = y + L * np.sin(theta_t)

        # Trailer: from hitch to trailer axle
        x_trailer = x - d * np.cos(theta_l)
        y_trailer = y - d * np.sin(theta_l)

        truck_line.set_data([x, x_front], [y, y_front])
        trailer_line.set_data([x, x_trailer], [y, y_trailer])
        truck_pts.set_data([x, x_front], [y, y_front])
        trailer_pts.set_data([x, x_trailer], [y, y_trailer])
        hitch_pt.set_data([x], [y])

        return truck_line, trailer_line, truck_pts, trailer_pts, hitch_pt

    anim = FuncAnimation(fig, update, init_func=init, frames=M,
                         interval=interval, blit=True, repeat=True)
    ax.legend([truck_line, trailer_line], ["Truck", "Trailer"],
              loc="upper right", fontsize=10)

    return anim

### Forward Path

In [ ]:
from IPython.display import HTML
params_forward_waypoints = {
    "L": 2.0, "d": 5.0,
    "q0": np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0]),

    "waypoints": [
        np.array([15.0, 0.0, 0.0, 0.0]),  
        np.array([30.0, 15.0, 0.0, 0.0]),   
        np.array([45.0, 15.0, 0.0, 0.0]), 
        np.array([45.0, 0.0, 0.0, 0.0]),  
    ],
    "waypoint_tol": 1.0, 

    # MPC parameters
    "dt": 0.1, "N": 15,
    "v_min": 0.0, "v_max": 3.0,
    "phi_min": -1.5, "phi_max": 1.5,
    "a_min": -2.0, "a_max": 2.0,
    "phi_dot_min": -1.0, "phi_dot_max": 1.0,
    "max_jackknife_angle": np.pi/2,
    "w_control": 0.01,
    "max_steps": 250,
    "solver_max_iter": 500,
    "solver_tol": 1e-6,
    "solver_print_level": 0,
    "a_init": 0.2,
    "phi_dot_init": 0.0
}

q_hist_wp_fwd, u_hist_wp_fwd, switches_fwd = run_mpc_loop(
    params_forward_waypoints)

plot_trajectory(q_hist_wp_fwd, u_hist_wp_fwd, params_forward_waypoints)

anim_wp_fwd = animate_waypoint_trajectory(
    q_hist_wp_fwd, params_forward_waypoints, switches_fwd, interval=60)
display(HTML(anim_wp_fwd.to_jshtml()))

### Reverse Path

In [ ]:
params_reverse_waypoints = {
    "L": 2.0, "d": 5.0,
    "q0": np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0]),

    "waypoints": [
        np.array([-10.0, 0.0, 0.0, 0.0]),   
        np.array([-20.0, -10.0, 0.0, 0.0]),  
        np.array([-30.0, -10.0, 0.0, 0.0]),  
    ],
    "waypoint_tol": 1.0,

    # MPC parameters
    "dt": 0.1, "N": 100,
    "v_min": -4.0, "v_max": 0.0,  # Reverse only
    "phi_min": -1.5, "phi_max": 1.5,
    "a_min": -2.0, "a_max": 2.0,
    "phi_dot_min": -0.5, "phi_dot_max": 0.5,
    "max_jackknife_angle": np.pi/2,
    "w_control": 0.01,
    "max_steps": 300,
    "solver_max_iter": 500,
    "solver_tol": 1e-6,
    "solver_print_level": 0,
    "a_init": -0.15,  
    "phi_dot_init": 0.0
}

q_hist_wp_rev, u_hist_wp_rev, switches_rev = run_mpc_loop(
    params_reverse_waypoints)

plot_trajectory(q_hist_wp_rev, u_hist_wp_rev, params_reverse_waypoints)

anim_wp_rev = animate_waypoint_trajectory(
    q_hist_wp_rev, params_reverse_waypoints, switches_rev, interval=60)
display(HTML(anim_wp_rev.to_jshtml()))